In [ ]:
!pip install pgmpy
import pgmpy
import pandas as pd
import numpy as np
from pgmpy.models import BayesianNetwork
from pgmpy.utils import get_example_model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 16.9 MB/s eta 0:00:00


In [ ]:
# Cargar la red ALARM desde el repositorio de ejemplos de pgmpy
# Esto define los Nodos y Arcos automáticamente con la estructura correcta médica
bn = get_example_model('child')

print("Nodos en la RB:", bn.nodes())
print("Número de nodos:", len(bn.nodes()))
print("Arcos en la RB:", bn.edges())
print("Número de arcos:", len(bn.edges()))

Nodos en la RB: ['BirthAsphyxia', 'HypDistrib', 'HypoxiaInO2', 'CO2', 'ChestXray', 'Grunting', 'LVHreport', 'LowerBodyO2', 'RUQO2', 'CO2Report', 'XrayReport', 'Disease', 'GruntingReport', 'Age', 'LVH', 'DuctFlow', 'CardiacMixing', 'LungParench', 'LungFlow', 'Sick']
Número de nodos: 20
Arcos en la RB: [('BirthAsphyxia', 'Disease'), ('HypDistrib', 'LowerBodyO2'), ('HypoxiaInO2', 'LowerBodyO2'), ('HypoxiaInO2', 'RUQO2'), ('CO2', 'CO2Report'), ('ChestXray', 'XrayReport'), ('Grunting', 'GruntingReport'), ('Disease', 'Age'), ('Disease', 'LVH'), ('Disease', 'DuctFlow'), ('Disease', 'CardiacMixing'), ('Disease', 'LungParench'), ('Disease', 'LungFlow'), ('Disease', 'Sick'), ('LVH', 'LVHreport'), ('DuctFlow', 'HypDistrib'), ('CardiacMixing', 'HypDistrib'), ('CardiacMixing', 'HypoxiaInO2'), ('LungParench', 'HypoxiaInO2'), ('LungParench', 'CO2'), ('LungParench', 'ChestXray'), ('LungParench', 'Grunting'), ('LungFlow', 'ChestXray'), ('Sick', 'Grunting'), ('Sick', 'Age')]
Número de arcos: 25


In [ ]:

from pgmpy.sampling import BayesianModelSampling

print("Generando dataset simulado...")
inference_sampler = BayesianModelSampling(bn)
data_child = inference_sampler.forward_sample(size=1000)

print(data_child.head())
print(data_child.info())

Generando dataset simulado...


  0%|          | 0/20 [00:00<?, ?it/s]

  BirthAsphyxia HypDistrib HypoxiaInO2     CO2  ChestXray Grunting LVHreport  \
0            no      Equal      Severe  Normal  Plethoric       no        no   
1            no      Equal      Severe  Normal  Plethoric       no        no   
2            no    Unequal    Moderate  Normal  Oligaemic       no        no   
3            no    Unequal      Severe  Normal  Oligaemic      yes        no   
4            no      Equal    Moderate  Normal     Normal       no       yes   

  LowerBodyO2 RUQO2 CO2Report XrayReport Disease GruntingReport       Age  \
0          <5  5-12     >=7.5  Plethoric     TGA             no  0-3_days   
1        5-12  5-12      <7.5  Plethoric     TGA             no  0-3_days   
2          <5    <5      <7.5  Oligaemic     PFC             no  0-3_days   
3          <5  5-12      <7.5  Oligaemic  Fallot            yes  0-3_days   
4         12+   12+      <7.5     Normal   PAIVS             no  0-3_days   

   LVH  DuctFlow CardiacMixing LungParench LungFlow Sick

In [ ]:
from pgmpy.estimators import MaximumLikelihoodEstimator

# Mostrar un ejemplo de CPD (Tablas de Probabilidad Condicional)
# Por ejemplo, para la presión sanguínea (BP)
for cpd in bn.get_cpds():
  print(cpd)

+-----------------+--------------+-----+---------------+---------------+
| Disease         | Disease(PFC) | ... | Disease(Lung) | Disease(Lung) |
+-----------------+--------------+-----+---------------+---------------+
| Sick            | Sick(yes)    | ... | Sick(yes)     | Sick(no)      |
+-----------------+--------------+-----+---------------+---------------+
| Age(0-3_days)   | 0.95         | ... | 0.9           | 0.8           |
+-----------------+--------------+-----+---------------+---------------+
| Age(4-10_days)  | 0.03         | ... | 0.08          | 0.15          |
+-----------------+--------------+-----+---------------+---------------+
| Age(11-30_days) | 0.02         | ... | 0.02          | 0.05          |
+-----------------+--------------+-----+---------------+---------------+
+--------------------+-----+
| BirthAsphyxia(yes) | 0.1 |
+--------------------+-----+
| BirthAsphyxia(no)  | 0.9 |
+--------------------+-----+
+-------------+---------------------+-----+---------

In [ ]:
import graphviz

# Crear el objeto Digraph
dot = graphviz.Digraph('BN_Child_Graph_Literal', comment='Bayesian Network Child Literal Tables', format='png')

# Configuración global del grafo
dot.attr(rankdir='TB')  # Top to Bottom
dot.attr('node', shape='plain') # Usamos 'plain' para que la tabla HTML defina el borde

# ==========================================
# DEFINICIÓN DE NODOS (TABLAS LITERALES)
# ==========================================

# 1. Age
# Estructura compleja: Disease y Sick como padres
label_age = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>Age</B></TD>
    <TD>Disease(PFC)</TD>
    <TD>...</TD>
    <TD>Disease(Lung)</TD>
    <TD>Disease(Lung)</TD>
  </TR>
  <TR>
    <TD>Sick</TD>
    <TD>Sick(yes)</TD>
    <TD>...</TD>
    <TD>Sick(yes)</TD>
    <TD>Sick(no)</TD>
  </TR>
  <TR>
    <TD>Age(0-3_days)</TD>
    <TD>0.95</TD>
    <TD>...</TD>
    <TD>0.9</TD>
    <TD>0.8</TD>
  </TR>
  <TR>
    <TD>Age(4-10_days)</TD>
    <TD>0.03</TD>
    <TD>...</TD>
    <TD>0.08</TD>
    <TD>0.15</TD>
  </TR>
  <TR>
    <TD>Age(11-30_days)</TD>
    <TD>0.02</TD>
    <TD>...</TD>
    <TD>0.02</TD>
    <TD>0.05</TD>
  </TR>
</TABLE>>'''
dot.node('Age', label=label_age)

# 2. BirthAsphyxia
label_ba = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR><TD>BirthAsphyxia(yes)</TD><TD>0.1</TD></TR>
  <TR><TD>BirthAsphyxia(no)</TD><TD>0.9</TD></TR>
</TABLE>>'''
dot.node('BirthAsphyxia', label=label_ba)

# 3. CO2
# Depende de LungParench
label_co2 = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>CO2</B></TD>
    <TD>LungParench(Normal)</TD>
    <TD>...</TD>
    <TD>LungParench(Abnormal)</TD>
  </TR>
  <TR>
    <TD>CO2(Normal)</TD>
    <TD>0.8</TD>
    <TD>...</TD>
    <TD>0.45</TD>
  </TR>
  <TR>
    <TD>CO2(Low)</TD>
    <TD>0.1</TD>
    <TD>...</TD>
    <TD>0.05</TD>
  </TR>
  <TR>
    <TD>CO2(High)</TD>
    <TD>0.1</TD>
    <TD>...</TD>
    <TD>0.5</TD>
  </TR>
</TABLE>>'''
dot.node('CO2', label=label_co2)

# 4. CO2Report
# Depende de CO2
label_co2r = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>CO2Report</B></TD>
    <TD>CO2(Normal)</TD>
    <TD>CO2(Low)</TD>
    <TD>CO2(High)</TD>
  </TR>
  <TR>
    <TD>CO2Report(&lt;7.5)</TD>
    <TD>0.9</TD>
    <TD>0.9</TD>
    <TD>0.1</TD>
  </TR>
  <TR>
    <TD>CO2Report(&gt;=7.5)</TD>
    <TD>0.1</TD>
    <TD>0.1</TD>
    <TD>0.9</TD>
  </TR>
</TABLE>>'''
dot.node('CO2Report', label=label_co2r)

# 5. CardiacMixing
# Depende de Disease
label_cm = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>CardiacMixing</B></TD>
    <TD>...</TD>
    <TD>Disease(TAPVD)</TD>
    <TD>Disease(Lung)</TD>
  </TR>
  <TR>
    <TD>CardiacMixing(None)</TD>
    <TD>...</TD>
    <TD>0.01</TD>
    <TD>0.4</TD>
  </TR>
  <TR>
    <TD>CardiacMixing(Mild)</TD>
    <TD>...</TD>
    <TD>0.03</TD>
    <TD>0.53</TD>
  </TR>
  <TR>
    <TD>CardiacMixing(Complete)</TD>
    <TD>...</TD>
    <TD>0.95</TD>
    <TD>0.05</TD>
  </TR>
  <TR>
    <TD>CardiacMixing(Transp.)</TD>
    <TD>...</TD>
    <TD>0.01</TD>
    <TD>0.02</TD>
  </TR>
</TABLE>>'''
dot.node('CardiacMixing', label=label_cm)

# 6. ChestXray
# Depende de LungParench y LungFlow
label_cx = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>ChestXray</B></TD>
    <TD>...</TD>
    <TD>LungParench(Abnormal)</TD>
  </TR>
  <TR>
    <TD>LungFlow</TD>
    <TD>...</TD>
    <TD>LungFlow(High)</TD>
  </TR>
  <TR>
    <TD>ChestXray(Normal)</TD>
    <TD>...</TD>
    <TD>0.24</TD>
  </TR>
  <TR>
    <TD>ChestXray(Oligaemic)</TD>
    <TD>...</TD>
    <TD>0.33</TD>
  </TR>
  <TR>
    <TD>ChestXray(Plethoric)</TD>
    <TD>...</TD>
    <TD>0.03</TD>
  </TR>
  <TR>
    <TD>ChestXray(Grd_Glass)</TD>
    <TD>...</TD>
    <TD>0.34</TD>
  </TR>
  <TR>
    <TD>ChestXray(Asy/Patch)</TD>
    <TD>...</TD>
    <TD>0.06</TD>
  </TR>
</TABLE>>'''
dot.node('ChestXray', label=label_cx)

# 7. Disease
# Depende de BirthAsphyxia
label_dis = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>Disease</B></TD>
    <TD>BirthAsphyxia(yes)</TD>
    <TD>BirthAsphyxia(no)</TD>
  </TR>
  <TR>
    <TD>Disease(PFC)</TD>
    <TD>0.2</TD>
    <TD>0.03061224</TD>
  </TR>
  <TR>
    <TD>Disease(TGA)</TD>
    <TD>0.3</TD>
    <TD>0.33673469</TD>
  </TR>
  <TR>
    <TD>Disease(Fallot)</TD>
    <TD>0.25</TD>
    <TD>0.29591837</TD>
  </TR>
  <TR>
    <TD>Disease(PAIVS)</TD>
    <TD>0.15</TD>
    <TD>0.23469388</TD>
  </TR>
  <TR>
    <TD>Disease(TAPVD)</TD>
    <TD>0.05</TD>
    <TD>0.05102041</TD>
  </TR>
  <TR>
    <TD>Disease(Lung)</TD>
    <TD>0.05</TD>
    <TD>0.05102041</TD>
  </TR>
</TABLE>>'''
dot.node('Disease', label=label_dis)

# 8. DuctFlow
# Depende de Disease
label_df = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>DuctFlow</B></TD>
    <TD>Disease(PFC)</TD>
    <TD>...</TD>
    <TD>Disease(TAPVD)</TD>
    <TD>Disease(Lung)</TD>
  </TR>
  <TR>
    <TD>DuctFlow(Lt_to_Rt)</TD>
    <TD>0.15</TD>
    <TD>...</TD>
    <TD>0.33</TD>
    <TD>0.2</TD>
  </TR>
  <TR>
    <TD>DuctFlow(None)</TD>
    <TD>0.05</TD>
    <TD>...</TD>
    <TD>0.33</TD>
    <TD>0.4</TD>
  </TR>
  <TR>
    <TD>DuctFlow(Rt_to_Lt)</TD>
    <TD>0.8</TD>
    <TD>...</TD>
    <TD>0.34</TD>
    <TD>0.4</TD>
  </TR>
</TABLE>>'''
dot.node('DuctFlow', label=label_df)

# 9. Grunting
# Depende de LungParench y Sick
label_gru = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>Grunting</B></TD>
    <TD>...</TD>
    <TD>LungParench(Abnormal)</TD>
  </TR>
  <TR>
    <TD>Sick</TD>
    <TD>...</TD>
    <TD>Sick(no)</TD>
  </TR>
  <TR>
    <TD>Grunting(yes)</TD>
    <TD>...</TD>
    <TD>0.6</TD>
  </TR>
  <TR>
    <TD>Grunting(no)</TD>
    <TD>...</TD>
    <TD>0.4</TD>
  </TR>
</TABLE>>'''
dot.node('Grunting', label=label_gru)

# 10. GruntingReport
# Depende de Grunting
label_grur = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>GruntingReport</B></TD>
    <TD>Grunting(yes)</TD>
    <TD>Grunting(no)</TD>
  </TR>
  <TR>
    <TD>GruntingReport(yes)</TD>
    <TD>0.8</TD>
    <TD>0.1</TD>
  </TR>
  <TR>
    <TD>GruntingReport(no)</TD>
    <TD>0.2</TD>
    <TD>0.9</TD>
  </TR>
</TABLE>>'''
dot.node('GruntingReport', label=label_grur)

# 11. HypDistrib
# Depende de DuctFlow y CardiacMixing
label_hd = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>HypDistrib</B></TD>
    <TD>...</TD>
    <TD>DuctFlow(Rt_to_Lt)</TD>
  </TR>
  <TR>
    <TD>CardiacMixing</TD>
    <TD>...</TD>
    <TD>CardiacMixing(Transp.)</TD>
  </TR>
  <TR>
    <TD>HypDistrib(Equal)</TD>
    <TD>...</TD>
    <TD>0.5</TD>
  </TR>
  <TR>
    <TD>HypDistrib(Unequal)</TD>
    <TD>...</TD>
    <TD>0.5</TD>
  </TR>
</TABLE>>'''
dot.node('HypDistrib', label=label_hd)

# 12. HypoxiaInO2
# Depende de CardiacMixing y LungParench
label_ho2 = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>HypoxiaInO2</B></TD>
    <TD>...</TD>
    <TD>CardiacMixing(Transp.)</TD>
  </TR>
  <TR>
    <TD>LungParench</TD>
    <TD>...</TD>
    <TD>LungParench(Abnormal)</TD>
  </TR>
  <TR>
    <TD>HypoxiaInO2(Mild)</TD>
    <TD>...</TD>
    <TD>0.02</TD>
  </TR>
  <TR>
    <TD>HypoxiaInO2(Moderate)</TD>
    <TD>...</TD>
    <TD>0.18</TD>
  </TR>
  <TR>
    <TD>HypoxiaInO2(Severe)</TD>
    <TD>...</TD>
    <TD>0.8</TD>
  </TR>
</TABLE>>'''
dot.node('HypoxiaInO2', label=label_ho2)

# 13. LVH
# Depende de Disease
label_lvh = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>LVH</B></TD>
    <TD>Disease(PFC)</TD>
    <TD>...</TD>
    <TD>Disease(TAPVD)</TD>
    <TD>Disease(Lung)</TD>
  </TR>
  <TR>
    <TD>LVH(yes)</TD>
    <TD>0.1</TD>
    <TD>...</TD>
    <TD>0.05</TD>
    <TD>0.1</TD>
  </TR>
  <TR>
    <TD>LVH(no)</TD>
    <TD>0.9</TD>
    <TD>...</TD>
    <TD>0.95</TD>
    <TD>0.9</TD>
  </TR>
</TABLE>>'''
dot.node('LVH', label=label_lvh)

# 14. LVHreport
# Depende de LVH
label_lvhr = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>LVHreport</B></TD>
    <TD>LVH(yes)</TD>
    <TD>LVH(no)</TD>
  </TR>
  <TR>
    <TD>LVHreport(yes)</TD>
    <TD>0.9</TD>
    <TD>0.05</TD>
  </TR>
  <TR>
    <TD>LVHreport(no)</TD>
    <TD>0.1</TD>
    <TD>0.95</TD>
  </TR>
</TABLE>>'''
dot.node('LVHreport', label=label_lvhr)

# 15. LowerBodyO2
# Depende de HypDistrib y HypoxiaInO2
label_lbo2 = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>LowerBodyO2</B></TD>
    <TD>...</TD>
    <TD>HypDistrib(Unequal)</TD>
  </TR>
  <TR>
    <TD>HypoxiaInO2</TD>
    <TD>...</TD>
    <TD>HypoxiaInO2(Severe)</TD>
  </TR>
  <TR>
    <TD>LowerBodyO2(&lt;5)</TD>
    <TD>...</TD>
    <TD>0.6</TD>
  </TR>
  <TR>
    <TD>LowerBodyO2(5-12)</TD>
    <TD>...</TD>
    <TD>0.35</TD>
  </TR>
  <TR>
    <TD>LowerBodyO2(12+)</TD>
    <TD>...</TD>
    <TD>0.05</TD>
  </TR>
</TABLE>>'''
dot.node('LowerBodyO2', label=label_lbo2)

# 16. LungFlow
# Depende de Disease
label_lf = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>LungFlow</B></TD>
    <TD>Disease(PFC)</TD>
    <TD>...</TD>
    <TD>Disease(TAPVD)</TD>
    <TD>Disease(Lung)</TD>
  </TR>
  <TR>
    <TD>LungFlow(Normal)</TD>
    <TD>0.3</TD>
    <TD>...</TD>
    <TD>0.3</TD>
    <TD>0.7</TD>
  </TR>
  <TR>
    <TD>LungFlow(Low)</TD>
    <TD>0.65</TD>
    <TD>...</TD>
    <TD>0.1</TD>
    <TD>0.1</TD>
  </TR>
  <TR>
    <TD>LungFlow(High)</TD>
    <TD>0.05</TD>
    <TD>...</TD>
    <TD>0.6</TD>
    <TD>0.2</TD>
  </TR>
</TABLE>>'''
dot.node('LungFlow', label=label_lf)

# 17. LungParench (Nodo CPT)
# Depende de Disease
label_lp = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>LungParench</B></TD>
    <TD>...</TD>
    <TD>Disease(TAPVD)</TD>
    <TD>Disease(Lung)</TD>
  </TR>
  <TR>
    <TD>LungParench(Normal)</TD>
    <TD>...</TD>
    <TD>0.1</TD>
    <TD>0.03</TD>
  </TR>
  <TR>
    <TD>LungParench(Congested)</TD>
    <TD>...</TD>
    <TD>0.6</TD>
    <TD>0.25</TD>
  </TR>
  <TR>
    <TD>LungParench(Abnormal)</TD>
    <TD>...</TD>
    <TD>0.3</TD>
    <TD>0.72</TD>
  </TR>
</TABLE>>'''
dot.node('LungParench', label=label_lp)

# 18. RUQO2
# Depende de HypoxiaInO2
label_ruq = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>RUQO2</B></TD>
    <TD>HypoxiaInO2(Mild)</TD>
    <TD>...</TD>
    <TD>HypoxiaInO2(Severe)</TD>
  </TR>
  <TR>
    <TD>RUQO2(&lt;5)</TD>
    <TD>0.1</TD>
    <TD>...</TD>
    <TD>0.5</TD>
  </TR>
  <TR>
    <TD>RUQO2(5-12)</TD>
    <TD>0.3</TD>
    <TD>...</TD>
    <TD>0.4</TD>
  </TR>
  <TR>
    <TD>RUQO2(12+)</TD>
    <TD>0.6</TD>
    <TD>...</TD>
    <TD>0.1</TD>
  </TR>
</TABLE>>'''
dot.node('RUQO2', label=label_ruq)

# 19. Sick
# Depende de Disease
label_sick = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>Sick</B></TD>
    <TD>Disease(PFC)</TD>
    <TD>...</TD>
    <TD>Disease(TAPVD)</TD>
    <TD>Disease(Lung)</TD>
  </TR>
  <TR>
    <TD>Sick(yes)</TD>
    <TD>0.4</TD>
    <TD>...</TD>
    <TD>0.7</TD>
    <TD>0.7</TD>
  </TR>
  <TR>
    <TD>Sick(no)</TD>
    <TD>0.6</TD>
    <TD>...</TD>
    <TD>0.3</TD>
    <TD>0.3</TD>
  </TR>
</TABLE>>'''
dot.node('Sick', label=label_sick)

# 20. XrayReport
# Depende de ChestXray
label_xr = '''<<TABLE BORDER="1" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4">
  <TR>
    <TD BGCOLOR="lightgrey"><B>XrayReport</B></TD>
    <TD>...</TD>
    <TD>ChestXray(Asy/Patch)</TD>
  </TR>
  <TR>
    <TD>XrayReport(Normal)</TD>
    <TD>...</TD>
    <TD>0.08</TD>
  </TR>
  <TR>
    <TD>XrayReport(Oligaemic)</TD>
    <TD>...</TD>
    <TD>0.02</TD>
  </TR>
  <TR>
    <TD>XrayReport(Plethoric)</TD>
    <TD>...</TD>
    <TD>0.1</TD>
  </TR>
  <TR>
    <TD>XrayReport(Grd_Glass)</TD>
    <TD>...</TD>
    <TD>0.1</TD>
  </TR>
  <TR>
    <TD>XrayReport(Asy/Patchy)</TD>
    <TD>...</TD>
    <TD>0.7</TD>
  </TR>
</TABLE>>'''
dot.node('XrayReport', label=label_xr)

# ==========================================
# DEFINICIÓN DE ARISTAS (RELACIONES PADRE-HIJO)
# ==========================================

dot.edge('BirthAsphyxia', 'Disease')

# Hijos de Disease
dot.edge('Disease', 'Age')
dot.edge('Disease', 'CardiacMixing')
dot.edge('Disease', 'DuctFlow')
dot.edge('Disease', 'LVH')
dot.edge('Disease', 'LungFlow')
dot.edge('Disease', 'LungParench')
dot.edge('Disease', 'Sick')

# Hijos de Sick
dot.edge('Sick', 'Age')
dot.edge('Sick', 'Grunting')

# Hijos de LungParench
dot.edge('LungParench', 'CO2')
dot.edge('LungParench', 'ChestXray')
dot.edge('LungParench', 'Grunting')
dot.edge('LungParench', 'HypoxiaInO2')

# Hijos de LungFlow
dot.edge('LungFlow', 'ChestXray')

# Hijos de CO2
dot.edge('CO2', 'CO2Report')

# Hijos de CardiacMixing
dot.edge('CardiacMixing', 'HypDistrib')
dot.edge('CardiacMixing', 'HypoxiaInO2')

# Hijos de DuctFlow
dot.edge('DuctFlow', 'HypDistrib')

# Hijos de Grunting
dot.edge('Grunting', 'GruntingReport')

# Hijos de HypDistrib
dot.edge('HypDistrib', 'LowerBodyO2')

# Hijos de HypoxiaInO2
dot.edge('HypoxiaInO2', 'LowerBodyO2')
dot.edge('HypoxiaInO2', 'RUQO2')

# Hijos de LVH
dot.edge('LVH', 'LVHreport')

# Hijos de ChestXray
dot.edge('ChestXray', 'XrayReport')

# ==========================================
# RENDERIZADO
# ==========================================
print("Generando grafo 'bn_child_literal.png'...")
dot.render('bn_child_literal', view=True, cleanup=True)
print("¡Hecho!")

Generando grafo 'bn_child_literal.png'...
¡Hecho!


In [ ]:
import graphviz

# Crear el objeto Digraph
dot = graphviz.Digraph('BN_Child_Graph_Complete', comment='Bayesian Network Child Complete', format='png')

# Configuración global
dot.attr(rankdir='TB')
dot.attr('node', shape='plain')

# --- 1. Age ---
# Tabla compleja con 2 padres: Disease y Sick
# Datos: Disease(PFC)|Sick(yes) -> 0.95
#        Disease(Lung)|Sick(yes) -> 0.9
#        Disease(Lung)|Sick(no)  -> 0.8
label_age = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>Age</B></TD><TD COLSPAN="4">CPT</TD></TR>
  <TR><TD>Disease</TD><TD>PFC</TD><TD>...</TD><TD COLSPAN="2">Lung</TD></TR>
  <TR><TD>Sick</TD><TD>yes</TD><TD>...</TD><TD>yes</TD><TD>no</TD></TR>
  <TR><TD>Age(0-3_days)</TD><TD>0.95</TD><TD>...</TD><TD>0.9</TD><TD>0.8</TD></TR>
  <TR><TD>Age(4-10_days)</TD><TD>0.03</TD><TD>...</TD><TD>0.08</TD><TD>0.15</TD></TR>
  <TR><TD>Age(11-30_days)</TD><TD>0.02</TD><TD>...</TD><TD>0.02</TD><TD>0.05</TD></TR>
</TABLE>>'''
dot.node('Age', label=label_age)

# --- 2. BirthAsphyxia ---
label_ba = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>BirthAsphyxia</B></TD><TD>Prob</TD></TR>
  <TR><TD>yes</TD><TD>0.1</TD></TR>
  <TR><TD>no</TD><TD>0.9</TD></TR>
</TABLE>>'''
dot.node('BirthAsphyxia', label=label_ba)

# --- 3. CO2 ---
# Parent: LungParench (Normal ... Abnormal)
label_co2 = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>CO2</B></TD><TD COLSPAN="3">CPT</TD></TR>
  <TR><TD>LungParench</TD><TD>Normal</TD><TD>...</TD><TD>Abnormal</TD></TR>
  <TR><TD>CO2(Normal)</TD><TD>0.8</TD><TD>...</TD><TD>0.45</TD></TR>
  <TR><TD>CO2(Low)</TD><TD>0.1</TD><TD>...</TD><TD>0.05</TD></TR>
  <TR><TD>CO2(High)</TD><TD>0.1</TD><TD>...</TD><TD>0.5</TD></TR>
</TABLE>>'''
dot.node('CO2', label=label_co2)

# --- 4. CO2Report ---
# Parent: CO2 (Normal, Low, High)
label_co2r = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>CO2Report</B></TD><TD COLSPAN="3">CPT</TD></TR>
  <TR><TD>CO2</TD><TD>Normal</TD><TD>Low</TD><TD>High</TD></TR>
  <TR><TD>Report(&lt;7.5)</TD><TD>0.9</TD><TD>0.9</TD><TD>0.1</TD></TR>
  <TR><TD>Report(&gt;=7.5)</TD><TD>0.1</TD><TD>0.1</TD><TD>0.9</TD></TR>
</TABLE>>'''
dot.node('CO2Report', label=label_co2r)

# --- 5. CardiacMixing ---
# Parent: Disease (... TAPVD, Lung)
label_cm = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>CardiacMixing</B></TD><TD COLSPAN="3">CPT</TD></TR>
  <TR><TD>Disease</TD><TD>...</TD><TD>TAPVD</TD><TD>Lung</TD></TR>
  <TR><TD>Mixing(None)</TD><TD>...</TD><TD>0.01</TD><TD>0.4</TD></TR>
  <TR><TD>Mixing(Mild)</TD><TD>...</TD><TD>0.03</TD><TD>0.53</TD></TR>
  <TR><TD>Mixing(Complete)</TD><TD>...</TD><TD>0.95</TD><TD>0.05</TD></TR>
  <TR><TD>Mixing(Transp.)</TD><TD>...</TD><TD>0.01</TD><TD>0.02</TD></TR>
</TABLE>>'''
dot.node('CardiacMixing', label=label_cm)

# --- 6. ChestXray ---
# Parents: LungParench, LungFlow
# Mostrando columnas clave: LP(Abnormal) y LF(High)
label_cx = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>ChestXray</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>LungParench</TD><TD>...</TD><TD>Abnormal</TD></TR>
  <TR><TD>LungFlow</TD><TD>...</TD><TD>High</TD></TR>
  <TR><TD>Xray(Normal)</TD><TD>...</TD><TD>0.24</TD></TR>
  <TR><TD>Xray(Oligaemic)</TD><TD>...</TD><TD>0.33</TD></TR>
  <TR><TD>Xray(Plethoric)</TD><TD>...</TD><TD>0.03</TD></TR>
  <TR><TD>Xray(Grd_Glass)</TD><TD>...</TD><TD>0.34</TD></TR>
  <TR><TD>Xray(Asy/Patch)</TD><TD>...</TD><TD>0.06</TD></TR>
</TABLE>>'''
dot.node('ChestXray', label=label_cx)

# --- 7. Disease ---
# Parent: BirthAsphyxia (yes, no)
label_dis = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>Disease</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>BirthAsphyxia</TD><TD>yes</TD><TD>no</TD></TR>
  <TR><TD>Disease(PFC)</TD><TD>0.2</TD><TD>0.03061224</TD></TR>
  <TR><TD>Disease(TGA)</TD><TD>0.3</TD><TD>0.33673469</TD></TR>
  <TR><TD>Disease(Fallot)</TD><TD>0.25</TD><TD>0.29591837</TD></TR>
  <TR><TD>Disease(PAIVS)</TD><TD>0.15</TD><TD>0.23469388</TD></TR>
  <TR><TD>Disease(TAPVD)</TD><TD>0.05</TD><TD>0.05102041</TD></TR>
  <TR><TD>Disease(Lung)</TD><TD>0.05</TD><TD>0.05102041</TD></TR>
</TABLE>>'''
dot.node('Disease', label=label_dis)

# --- 8. DuctFlow ---
# Parent: Disease
label_df = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>DuctFlow</B></TD><TD COLSPAN="4">CPT</TD></TR>
  <TR><TD>Disease</TD><TD>PFC</TD><TD>...</TD><TD>TAPVD</TD><TD>Lung</TD></TR>
  <TR><TD>Lt_to_Rt</TD><TD>0.15</TD><TD>...</TD><TD>0.33</TD><TD>0.2</TD></TR>
  <TR><TD>None</TD><TD>0.05</TD><TD>...</TD><TD>0.33</TD><TD>0.4</TD></TR>
  <TR><TD>Rt_to_Lt</TD><TD>0.8</TD><TD>...</TD><TD>0.34</TD><TD>0.4</TD></TR>
</TABLE>>'''
dot.node('DuctFlow', label=label_df)

# --- 9. Grunting ---
# Parents: LungParench, Sick
label_gru = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>Grunting</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>LungParench</TD><TD>...</TD><TD>Abnormal</TD></TR>
  <TR><TD>Sick</TD><TD>...</TD><TD>no</TD></TR>
  <TR><TD>Grunting(yes)</TD><TD>...</TD><TD>0.6</TD></TR>
  <TR><TD>Grunting(no)</TD><TD>...</TD><TD>0.4</TD></TR>
</TABLE>>'''
dot.node('Grunting', label=label_gru)

# --- 10. GruntingReport ---
label_grur = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>GruntingReport</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>Grunting</TD><TD>yes</TD><TD>no</TD></TR>
  <TR><TD>Report(yes)</TD><TD>0.8</TD><TD>0.1</TD></TR>
  <TR><TD>Report(no)</TD><TD>0.2</TD><TD>0.9</TD></TR>
</TABLE>>'''
dot.node('GruntingReport', label=label_grur)

# --- 11. HypDistrib ---
# Parents: DuctFlow, CardiacMixing
label_hd = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>HypDistrib</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>DuctFlow</TD><TD>...</TD><TD>Rt_to_Lt</TD></TR>
  <TR><TD>CardiacMixing</TD><TD>...</TD><TD>Transp.</TD></TR>
  <TR><TD>Equal</TD><TD>...</TD><TD>0.5</TD></TR>
  <TR><TD>Unequal</TD><TD>...</TD><TD>0.5</TD></TR>
</TABLE>>'''
dot.node('HypDistrib', label=label_hd)

# --- 12. HypoxiaInO2 ---
# Parents: CardiacMixing, LungParench
label_ho2 = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>HypoxiaInO2</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>CardiacMixing</TD><TD>...</TD><TD>Transp.</TD></TR>
  <TR><TD>LungParench</TD><TD>...</TD><TD>Abnormal</TD></TR>
  <TR><TD>Hypoxia(Mild)</TD><TD>...</TD><TD>0.02</TD></TR>
  <TR><TD>Hypoxia(Mod)</TD><TD>...</TD><TD>0.18</TD></TR>
  <TR><TD>Hypoxia(Sev)</TD><TD>...</TD><TD>0.8</TD></TR>
</TABLE>>'''
dot.node('HypoxiaInO2', label=label_ho2)

# --- 13. LVH ---
label_lvh = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>LVH</B></TD><TD COLSPAN="4">CPT</TD></TR>
  <TR><TD>Disease</TD><TD>PFC</TD><TD>...</TD><TD>TAPVD</TD><TD>Lung</TD></TR>
  <TR><TD>LVH(yes)</TD><TD>0.1</TD><TD>...</TD><TD>0.05</TD><TD>0.1</TD></TR>
  <TR><TD>LVH(no)</TD><TD>0.9</TD><TD>...</TD><TD>0.95</TD><TD>0.9</TD></TR>
</TABLE>>'''
dot.node('LVH', label=label_lvh)

# --- 14. LVHreport ---
label_lvhr = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>LVHreport</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>LVH</TD><TD>yes</TD><TD>no</TD></TR>
  <TR><TD>Report(yes)</TD><TD>0.9</TD><TD>0.05</TD></TR>
  <TR><TD>Report(no)</TD><TD>0.1</TD><TD>0.95</TD></TR>
</TABLE>>'''
dot.node('LVHreport', label=label_lvhr)

# --- 15. LowerBodyO2 ---
# Parents: HypDistrib, HypoxiaInO2
label_lbo2 = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>LowerBodyO2</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>HypDistrib</TD><TD>...</TD><TD>Unequal</TD></TR>
  <TR><TD>HypoxiaInO2</TD><TD>...</TD><TD>Severe</TD></TR>
  <TR><TD>LowerBodyO2(&lt;5)</TD><TD>...</TD><TD>0.6</TD></TR>
  <TR><TD>LowerBodyO2(5-12)</TD><TD>...</TD><TD>0.35</TD></TR>
  <TR><TD>LowerBodyO2(12+)</TD><TD>...</TD><TD>0.05</TD></TR>
</TABLE>>'''
dot.node('LowerBodyO2', label=label_lbo2)

# --- 16. LungFlow ---
label_lf = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>LungFlow</B></TD><TD COLSPAN="4">CPT</TD></TR>
  <TR><TD>Disease</TD><TD>PFC</TD><TD>...</TD><TD>TAPVD</TD><TD>Lung</TD></TR>
  <TR><TD>Normal</TD><TD>0.3</TD><TD>...</TD><TD>0.3</TD><TD>0.7</TD></TR>
  <TR><TD>Low</TD><TD>0.65</TD><TD>...</TD><TD>0.1</TD><TD>0.1</TD></TR>
  <TR><TD>High</TD><TD>0.05</TD><TD>...</TD><TD>0.6</TD><TD>0.2</TD></TR>
</TABLE>>'''
dot.node('LungFlow', label=label_lf)

# --- 17. LungParench ---
label_lp = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>LungParench</B></TD><TD COLSPAN="3">CPT</TD></TR>
  <TR><TD>Disease</TD><TD>...</TD><TD>TAPVD</TD><TD>Lung</TD></TR>
  <TR><TD>Normal</TD><TD>...</TD><TD>0.1</TD><TD>0.03</TD></TR>
  <TR><TD>Congested</TD><TD>...</TD><TD>0.6</TD><TD>0.25</TD></TR>
  <TR><TD>Abnormal</TD><TD>...</TD><TD>0.3</TD><TD>0.72</TD></TR>
</TABLE>>'''
dot.node('LungParench', label=label_lp)

# --- 18. RUQO2 ---
# Parent: HypoxiaInO2 (Mild ... Severe)
label_ruq = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>RUQO2</B></TD><TD COLSPAN="3">CPT</TD></TR>
  <TR><TD>HypoxiaInO2</TD><TD>Mild</TD><TD>...</TD><TD>Severe</TD></TR>
  <TR><TD>RUQO2(&lt;5)</TD><TD>0.1</TD><TD>...</TD><TD>0.5</TD></TR>
  <TR><TD>RUQO2(5-12)</TD><TD>0.3</TD><TD>...</TD><TD>0.4</TD></TR>
  <TR><TD>RUQO2(12+)</TD><TD>0.6</TD><TD>...</TD><TD>0.1</TD></TR>
</TABLE>>'''
dot.node('RUQO2', label=label_ruq)

# --- 19. Sick ---
label_sick = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>Sick</B></TD><TD COLSPAN="4">CPT</TD></TR>
  <TR><TD>Disease</TD><TD>PFC</TD><TD>...</TD><TD>TAPVD</TD><TD>Lung</TD></TR>
  <TR><TD>Sick(yes)</TD><TD>0.4</TD><TD>...</TD><TD>0.7</TD><TD>0.7</TD></TR>
  <TR><TD>Sick(no)</TD><TD>0.6</TD><TD>...</TD><TD>0.3</TD><TD>0.3</TD></TR>
</TABLE>>'''
dot.node('Sick', label=label_sick)

# --- 20. XrayReport ---
label_xr = '''<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0">
  <TR><TD BGCOLOR="lightgrey"><B>XrayReport</B></TD><TD COLSPAN="2">CPT</TD></TR>
  <TR><TD>ChestXray</TD><TD>...</TD><TD>Asy/Patch</TD></TR>
  <TR><TD>Report(Normal)</TD><TD>...</TD><TD>0.08</TD></TR>
  <TR><TD>Report(Oligaemic)</TD><TD>...</TD><TD>0.02</TD></TR>
  <TR><TD>Report(Plethoric)</TD><TD>...</TD><TD>0.1</TD></TR>
  <TR><TD>Report(Grd_Glass)</TD><TD>...</TD><TD>0.1</TD></TR>
  <TR><TD>Report(Asy/Patchy)</TD><TD>...</TD><TD>0.7</TD></TR>
</TABLE>>'''
dot.node('XrayReport', label=label_xr)

# --- Definición de ARISTAS ---
dot.edge('BirthAsphyxia', 'Disease')
dot.edge('Disease', 'Age')
dot.edge('Disease', 'CardiacMixing')
dot.edge('Disease', 'DuctFlow')
dot.edge('Disease', 'LVH')
dot.edge('Disease', 'LungFlow')
dot.edge('Disease', 'LungParench')
dot.edge('Disease', 'Sick')
dot.edge('Sick', 'Age')
dot.edge('Sick', 'Grunting')
dot.edge('LungParench', 'CO2')
dot.edge('LungParench', 'ChestXray')
dot.edge('LungParench', 'Grunting')
dot.edge('LungParench', 'HypoxiaInO2')
dot.edge('LungFlow', 'ChestXray')
dot.edge('CO2', 'CO2Report')
dot.edge('CardiacMixing', 'HypDistrib')
dot.edge('CardiacMixing', 'HypoxiaInO2')
dot.edge('DuctFlow', 'HypDistrib')
dot.edge('Grunting', 'GruntingReport')
dot.edge('HypDistrib', 'LowerBodyO2')
dot.edge('HypoxiaInO2', 'LowerBodyO2')
dot.edge('HypoxiaInO2', 'RUQO2')
dot.edge('LVH', 'LVHreport')
dot.edge('ChestXray', 'XrayReport')

# Renderizar
print("Generando grafo 'bn_child_graph_complete.png'...")
dot.render('bn_child_graph_complete', view=True, cleanup=True)
print("¡Hecho!")

Generando grafo 'bn_child_graph_complete.png'...
¡Hecho!


In [ ]:
#This code allow us to compute all independencies
from pgmpy.base import DAG
G = DAG()
G.add_nodes_from(bn.nodes())
G.add_edges_from(bn.edges())
G.get_independencies()

(CardiacMixing ⟂ RUQO2 | HypoxiaInO2)
(RUQO2 ⟂ GruntingReport | Grunting)
(XrayReport ⟂ HypDistrib | ChestXray)
(Age ⟂ RUQO2 | Disease)
(GruntingReport ⟂ BirthAsphyxia | Grunting)
(XrayReport ⟂ CO2Report | ChestXray)
(ChestXray ⟂ HypDistrib | DuctFlow, CardiacMixing)
(XrayReport ⟂ Grunting | ChestXray)
(ChestXray ⟂ Disease | LungParench, LungFlow)
(LungFlow ⟂ Sick | Disease)
(ChestXray ⟂ CO2 | LungParench)
(LungFlow ⟂ HypoxiaInO2 | Disease)
(HypoxiaInO2 ⟂ Sick | Disease)
(LungFlow ⟂ CO2Report | Disease)
(Grunting ⟂ RUQO2 | LungParench, Sick)
(XrayReport ⟂ LVHreport | ChestXray)
(ChestXray ⟂ HypoxiaInO2 | LungParench, CardiacMixing)
(LVH ⟂ CO2 | Disease)
(LVH ⟂ HypoxiaInO2 | Disease)
(LVH ⟂ GruntingReport | Grunting)
(DuctFlow ⟂ CO2Report | Disease)
(LVH ⟂ Sick | Disease)
(ChestXray ⟂ LVHreport | LungParench, LungFlow)
(Sick ⟂ LVHreport | Disease)
(Age ⟂ HypoxiaInO2 | Disease)
(LVH ⟂ BirthAsphyxia | Disease)
(LVHreport ⟂ BirthAsphyxia | LVH)
(ChestXray ⟂ BirthAsphyxia | LungParench, Lun

In [ ]:
print("--- CASO A: Conexión en Serie ---")
# ¿Están conectados Asfixia y Edad? Sí, a través de la Enfermedad.
print("1. ¿BirthAsphyxia y Age conectados sin evidencia?:",
      bn.is_dconnected('BirthAsphyxia', 'Age'))

# Si observamos 'Disease', se vuelven independientes (d-separados)
print("2. ¿BirthAsphyxia y Age conectados dado Disease?:",
      bn.is_dconnected('BirthAsphyxia', 'Age', observed=['Disease']))

--- CASO A: Conexión en Serie ---
1. ¿BirthAsphyxia y Age conectados sin evidencia?: True
2. ¿BirthAsphyxia y Age conectados dado Disease?: False


In [ ]:
print("\n--- CASO B: Estructura en V (Colisionador) ---")
# LungParench y CardiacMixing son causas independientes de Hipoxia.
print("1. ¿LungParench y CardiacMixing conectados sin evidencia?:",
      bn.is_dconnected('LungParench', 'CardiacMixing'))

# Si observamos el efecto común 'HypoxiaInO2', se crea un camino activo (Intercausalidad)
print("2. ¿LungParench y CardiacMixing conectados dado HypoxiaInO2?:",
      bn.is_dconnected('LungParench', 'CardiacMixing', observed=['HypoxiaInO2']))


--- CASO B: Estructura en V (Colisionador) ---
1. ¿LungParench y CardiacMixing conectados sin evidencia?: True
2. ¿LungParench y CardiacMixing conectados dado HypoxiaInO2?: True


In [ ]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(bn)

# --- 1. Razonamiento Diagnóstico (Efecto -> Causa) ---
# Vemos un reporte de Rayos X con "Plethoric" (exceso de sangre).
# ¿Cuál es la probabilidad de que la enfermedad sea TGA (Transposición de Grandes Arterias)?
print("\n--- Diagnóstico: Reporte de Rayos X -> Enfermedad ---")
q_base = infer.query(['Disease'], evidence={})
q_diag = infer.query(['Disease'], evidence={'XrayReport': 'Plethoric'})

print("Probabilidad de TGA (sin evidencia):", q_base.values[5]) # Indice 5 suele ser TGA, verificar print
print("Probabilidad de TGA dado Xray Plethoric:", q_diag.values[5])


# --- 2. Razonamiento Intercausal (Explaining Away) ---
# Fenómeno: "Explicar" una causa reduce la probabilidad de la otra.

# Nodo objetivo: CardiacMixing (Mezcla cardiaca).
# Estado de interés: 'Transp.' (Transposición).

print("\n--- Explaining Away (Intercausalidad) ---")

# Paso 1: Probabilidad base de Mezcla Cardíaca
prob_base = infer.query(['CardiacMixing']).values
print(f"1. Prob. base de CardiacMixing=Transp: {prob_base[3]:.4f}")
# (Asumimos índice 3 es Transp, verificar orden en tu salida)

# Paso 2: Observamos un efecto grave (Hipoxia Severa). Ambas explicaciones posibles suben.
prob_effect = infer.query(['CardiacMixing'], evidence={'HypoxiaInO2': 'Severe'}).values
print(f"2. Prob. tras ver Hipoxia Severa: {prob_effect[3]:.4f} (SUBE)")

# Paso 3: Descubrimos que la OTRA causa es cierta (El parénquima pulmonar está Congestionado).
# Esto 'explica' la hipoxia, así que la probabilidad de que sea culpa del corazón debería BAJAR.
prob_explained = infer.query(['CardiacMixing'],
                             evidence={'HypoxiaInO2': 'Severe', 'LungParench': 'Congested'}).values
print(f"3. Prob. tras confirmar causa pulmonar: {prob_explained[3]:.4f} (BAJA - Explained Away)")


--- Diagnóstico: Reporte de Rayos X -> Enfermedad ---
Probabilidad de TGA (sin evidencia): 0.050918369
Probabilidad de TGA dado Xray Plethoric: 0.0351573049009509

--- Explaining Away (Intercausalidad) ---
1. Prob. base de CardiacMixing=Transp: 0.2793
2. Prob. tras ver Hipoxia Severa: 0.5900 (SUBE)
3. Prob. tras confirmar causa pulmonar: 0.3169 (BAJA - Explained Away)


In [ ]:
# 2. Inicializar la inferencia
inference = VariableElimination(bn)

# --- ESCENARIO 1: Razonamiento Causal (Causa -> Efecto) ---
# Comparación equivalente a "MakeModel" influyendo en "MedCost"
# Aquí miramos cómo la 'Asfixia al nacer' influye en la 'Enfermedad' base.

print("--- Query 1: Probabilidad de Enfermedad SIN asfixia al nacer ---")
phi_query_1 = inference.query(variables=['Disease'], evidence={'BirthAsphyxia': 'no'})
print(phi_query_1)

print("\n--- Query 2: Probabilidad de Enfermedad CON asfixia al nacer ---")
phi_query_2 = inference.query(variables=['Disease'], evidence={'BirthAsphyxia': 'yes'})
print(phi_query_2)


# --- ESCENARIO 2: Razonamiento de Predicción de Síntomas ---
# Comparación equivalente a predecir "ThisCarCost" (un gasto específico) dado el modelo.
# Aquí miramos cómo la presencia de 'Grunting' (Quejido respiratorio) predice el 'Reporte de Rayos X'.

print("\n--- Query 3: Reporte de Rayos X si el bebé NO presenta quejido (Grunting) ---")
phi_query_3 = inference.query(variables=['XrayReport'], evidence={'Grunting': 'no'})
print(phi_query_3)

print("\n--- Query 4: Reporte de Rayos X si el bebé SÍ presenta quejido (Grunting) ---")
phi_query_4 = inference.query(variables=['XrayReport'], evidence={'Grunting': 'yes'})
print(phi_query_4)

--- Query 1: Probabilidad de Enfermedad SIN asfixia al nacer ---
+-----------------+----------------+
| Disease         |   phi(Disease) |
+=================+================+
| Disease(PFC)    |         0.0306 |
+-----------------+----------------+
| Disease(TGA)    |         0.3367 |
+-----------------+----------------+
| Disease(Fallot) |         0.2959 |
+-----------------+----------------+
| Disease(PAIVS)  |         0.2347 |
+-----------------+----------------+
| Disease(TAPVD)  |         0.0510 |
+-----------------+----------------+
| Disease(Lung)   |         0.0510 |
+-----------------+----------------+

--- Query 2: Probabilidad de Enfermedad CON asfixia al nacer ---
+-----------------+----------------+
| Disease         |   phi(Disease) |
+=================+================+
| Disease(PFC)    |         0.2000 |
+-----------------+----------------+
| Disease(TGA)    |         0.3000 |
+-----------------+----------------+
| Disease(Fallot) |         0.2500 |
+-----------------

In [ ]:
print("--- Explorando D-Separation en la red 'Child' ---\n")

# CASO 1: Bloqueo de un camino (Serial o Divergente)
# Camino: BirthAsphyxia -> Disease -> LungParench -> ChestXray
# Sin observar nada, la asfixia al nacer influye en los rayos X (Active Trail).
# Si observamos la enfermedad ('Disease') o 'LungParench', cortamos el flujo de información.

print(f"1. ¿Conectados sin observar nada?: {bn.is_dconnected('BirthAsphyxia', 'ChestXray')}")
# Output esperado: True

print(f"2. ¿Conectados observando 'Disease'?: {bn.is_dconnected('BirthAsphyxia', 'ChestXray', observed=['Disease'])}")
# Output esperado: False (D-Separation lograda)


# CASO 2: Activación de un camino (Estructura en V / Collider)
# Estructura: LungParench -> HypoxiaInO2 <- CardiacMixing
# Dos causas independientes (LungParench y CardiacMixing) producen un efecto común (HypoxiaInO2).
# Normalmente están desconectadas (independientes). Si observamos el efecto común, se vuelven dependientes ("explaining away").

print(f"3. ¿Conectados sin observar nada?: {bn.is_dconnected('LungParench', 'CardiacMixing', observed=[])}")
# Output esperado: False (Son independientes a priori)

print(f"4. ¿Conectados observando el efecto común ('HypoxiaInO2')?: {bn.is_dconnected('LungParench', 'CardiacMixing', observed=['HypoxiaInO2'])}")
# Output esperado: True (El camino se activa)

--- Explorando D-Separation en la red 'Child' ---

1. ¿Conectados sin observar nada?: True
2. ¿Conectados observando 'Disease'?: False
3. ¿Conectados sin observar nada?: True
4. ¿Conectados observando el efecto común ('HypoxiaInO2')?: True


In [ ]:
target_node = 'Disease'

print(f"--- Analizando Markov Blanket para: {target_node} ---\n")

# 1. Búsqueda de nodos d-conectados SIN observar nada
# Debería mostrar muchos nodos, ya que la enfermedad influye en casi todo.
print("1. Active trails (sin observaciones):")
print(bn.active_trail_nodes(target_node))
print("-" * 50)

# 2. Obtener la Manta de Markov (Markov Blanket)
# MB = Padres + Hijos + Padres de los hijos (Co-parents)
disease_mb = bn.get_markov_blanket(target_node)
print(f"Markov Blanket de '{target_node}': {disease_mb}")

# 3. Verificar d-conexión observando TODA la Manta de Markov
# Teoría: El resultado debería ser SOLO el propio nodo 'Disease'.
# La manta "blinda" al nodo del resto de la red.
result1 = bn.active_trail_nodes(target_node, observed=disease_mb)
print(f"\n2. Active trails observando la MB completa (debería estar aislado):")
print(result1)
print("-" * 50)

# 4. "Romper" la manta eliminando un nodo clave
# Vamos a quitar 'BirthAsphyxia' (que es padre de Disease).
# Al dejar de observarlo, se abre el camino hacia arriba.
disease_mb_broken = bn.get_markov_blanket(target_node) # Volvemos a pedirla para tener la lista original
node_to_remove = 'BirthAsphyxia'

if node_to_remove in disease_mb_broken:
    disease_mb_broken.remove(node_to_remove)
    print(f"\n3. Rompemos la manta eliminando: '{node_to_remove}'")
else:
    print(f"El nodo {node_to_remove} no estaba en la MB.")

# 5. Verificar d-conexión con la manta incompleta
result2 = bn.active_trail_nodes(target_node, observed=disease_mb_broken)
print(f"Active trails con MB incompleta:")
print(result2)

# 6. Mostrar qué nodos se "filtraron" (se volvieron a conectar)
# Hacemos la diferencia de conjuntos para ver qué nodos nuevos aparecieron
leaked_nodes = result2[target_node] - set(disease_mb_broken) - {target_node}
print(f"\nNodos que se volvieron a conectar (Fuga de información):")
print(leaked_nodes)

--- Analizando Markov Blanket para: Disease ---

1. Active trails (sin observaciones):
{'Disease': {'LungParench', 'ChestXray', 'Age', 'DuctFlow', 'LVH', 'LungFlow', 'HypoxiaInO2', 'XrayReport', 'Sick', 'Grunting', 'Disease', 'CO2Report', 'RUQO2', 'GruntingReport', 'LVHreport', 'CO2', 'CardiacMixing', 'BirthAsphyxia', 'HypDistrib', 'LowerBodyO2'}}
--------------------------------------------------
Markov Blanket de 'Disease': ['LungParench', 'Age', 'LVH', 'DuctFlow', 'LungFlow', 'BirthAsphyxia', 'CardiacMixing', 'Sick']

2. Active trails observando la MB completa (debería estar aislado):
{'Disease': {'Disease'}}
--------------------------------------------------

3. Rompemos la manta eliminando: 'BirthAsphyxia'
Active trails con MB incompleta:
{'Disease': {'Disease', 'BirthAsphyxia'}}

Nodos que se volvieron a conectar (Fuga de información):
{'BirthAsphyxia'}


In [ ]:
import warnings
import pandas as pd
import numpy as np
from pgmpy.inference import ApproxInference, VariableElimination

# 1. Silenciar advertencias no críticas
warnings.filterwarnings("ignore", category=FutureWarning) # Pandas
# Opcional: silenciar warnings de pgmpy si molestan mucho
# import logging
# logging.getLogger("pgmpy").setLevel(logging.ERROR)

print("\n=== COMPARATIVA ROBUSTA: INFERENCIA APROXIMADA VS EXACTA ===")

# Configuración
variables_interes = ['Disease']
evidencia = {'Grunting': 'yes', 'Age': '0-3_days'}
n_muestras = 10000

# --- EXACTA ---
ve_infer = VariableElimination(bn)
q_exact = ve_infer.query(variables=variables_interes, evidence=evidencia, show_progress=False)

# --- APROXIMADA ---
np.random.seed(42) # Semilla para consistencia
approx_infer = ApproxInference(bn)
q_approx = approx_infer.query(variables=variables_interes,
                              evidence=evidencia,
                              n_samples=n_muestras,
                              show_progress=True)

# --- COMPARACIÓN INTELIGENTE (Mapeo por Nombre) ---
print("\n" + "-"*65)
print(f"{'Estado':<10} | {'Exacto':<12} | {'Aproximado':<12} | {'Error Abs':<12}")
print("-" * 65)

# 1. Crear diccionarios {Nombre_Estado: Probabilidad}
# Esto asegura que no importa el orden en que vengan, siempre comparamos el correcto.
dict_exact = dict(zip(q_exact.state_names['Disease'], q_exact.values))
dict_approx = dict(zip(q_approx.state_names['Disease'], q_approx.values))

# 2. Iterar sobre los nombres de los estados
for estado in dict_exact.keys():
    p_exact = dict_exact[estado]

    # .get(estado, 0.0) maneja el caso raro donde el muestreo nunca encuentra ese estado
    p_approx = dict_approx.get(estado, 0.0)

    error = abs(p_exact - p_approx)

    # Imprimir resultado
    print(f"{estado:<10} | {p_exact:.4f}       | {p_approx:.4f}       | {error:.4f}")

print("-" * 65)


=== COMPARATIVA ROBUSTA: INFERENCIA APROXIMADA VS EXACTA ===


  0%|          | 0/10000 [00:00<?, ?it/s]


-----------------------------------------------------------------
Estado     | Exacto       | Aproximado   | Error Abs   
-----------------------------------------------------------------
PFC        | 0.0804       | 0.0814       | 0.0010
TGA        | 0.2939       | 0.2953       | 0.0014
Fallot     | 0.1280       | 0.1241       | 0.0039
PAIVS      | 0.2135       | 0.2186       | 0.0051
TAPVD      | 0.1099       | 0.1111       | 0.0012
Lung       | 0.1744       | 0.1695       | 0.0049
-----------------------------------------------------------------
